# Build the Master Patient Table

## Imports

In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(override=True)

## Load the Data

In [ ]:
base_path = Path(os.getenv("BASE_DIR"))

In [ ]:
pat_data = base_path/"patients.csv"
df_pat = pd.read_csv(pat_data)

In [ ]:
df_pat.head()

In [ ]:
df_pat.info()

In [ ]:
cond_data = base_path/"conditions.csv"
df_cond = pd.read_csv(cond_data)

In [ ]:
df_cond.head()

In [ ]:
df_cond.info()

In [ ]:
enc_data = base_path/"encounters.csv"
df_enc = pd.read_csv(enc_data)

In [ ]:
df_enc.head()

In [ ]:
df_enc.info()

In [ ]:
med_data = base_path/"medications.csv"
df_med = pd.read_csv(med_data)

In [ ]:
df_med.head()

In [ ]:
df_med.info()

In [ ]:
print("Patients:", df_pat.shape)
print("Conditions:", df_cond.shape)
print("Encounters:", df_enc.shape)
print("Medications:", df_med.shape)

## Calculate Patient Age

In [ ]:
# Create "AGE" column
df_pat["BIRTHDATE"] = pd.to_datetime(df_pat["BIRTHDATE"])
today = pd.Timestamp.today()
df_pat["AGE"] = (
    (today - df_pat["BIRTHDATE"]).dt.days // 365
)

In [ ]:
# Create a smaller demographics table
patient_demo = df_pat[
    [
        "Id",
        "AGE",
        "GENDER",
        "RACE",
        "ETHNICITY",
        "HEALTHCARE_EXPENSES",
        "HEALTHCARE_COVERAGE"
    ]
].copy()

patient_demo.rename(columns={"Id": "PATIENT"}, inplace=True)

patient_demo.head()

## Aggregate Conditions

In [ ]:
patient_conditions = (
    df_cond
    .groupby("PATIENT")["DESCRIPTION"]
    .apply(list)
    .reset_index()
)

patient_conditions.rename(
    columns={"DESCRIPTION": "CONDITIONS"},
    inplace=True
)

patient_conditions.head()

## Aggregate Medications

In [ ]:
patient_medications = (
    df_med
    .groupby("PATIENT")["DESCRIPTION"]
    .apply(list)
    .reset_index() 
)

patient_medications.rename(
    columns={"DESCRIPTION": "MEDICATIONS"},
    inplace=True
)

patient_medications.head()

## Encounter Count

In [ ]:
encounter_counts = (
    df_enc
    .groupby("PATIENT")
    .size()
    .reset_index(name="ENCOUNTER_COUNT")
)

encounter_counts.head()

## Merge Everything

In [ ]:
patient_df = (
    patient_demo
    .merge(patient_conditions, on="PATIENT", how="left")
    .merge(patient_medications, on="PATIENT", how="left")
    .merge(encounter_counts, on="PATIENT", how="left")
)

In [ ]:
# Fill missing values
patient_df["CONDITIONS"] = patient_df["CONDITIONS"].apply(
    lambda x: x if isinstance(x, list) else []
)

patient_df["MEDICATIONS"] = patient_df["MEDICATIONS"].apply(
    lambda x: x if isinstance(x, list) else []
)

patient_df["ENCOUNTER_COUNT"] = (
    patient_df["ENCOUNTER_COUNT"]
    .fillna(0)
    .astype(int)
)

In [ ]:
# Remove duplicates
def unique_list(items):
    return list(dict.fromkeys(items))

patient_df["CONDITIONS"] = patient_df["CONDITIONS"].apply(unique_list)
patient_df["MEDICATIONS"] = patient_df["MEDICATIONS"].apply(unique_list)

## Inspect the Final Dataset

In [ ]:
print(patient_df.shape)

patient_df.head()